# 02 · P&L y rentabilidad global

Construye la cuenta de resultados operativa y localiza la evolución de ingresos, EBITDA y margen.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Cuenta de resultados anual

In [2]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
annual=a.groupby('year',as_index=False).agg(revenue=('revenue','sum'),variable_cost=('variable_cost','sum'),fixed_cost=('fixed_allocated','sum'),ebitda=('ebitda','sum'))
annual['ebitda_margin']=annual.ebitda/annual.revenue
annual.to_csv(TABLES/'02_pyg_anual.csv',index=False)
display(annual)

   year        revenue  ...        ebitda  ebitda_margin
0  2024 181,789,537.62  ...  8,801,538.82           0.05
1  2025 192,205,457.88  ... 10,932,407.56           0.06
2  2026 200,111,488.76  ... 19,850,741.30           0.10

[3 rows x 6 columns]


## Tendencia mensual

In [3]:
monthly=a.groupby('month',as_index=False).agg(revenue=('revenue','sum'),ebitda=('ebitda','sum'))
monthly['ebitda_margin']=monthly.ebitda/monthly.revenue
fig,ax=plt.subplots(2,1,sharex=True,figsize=(12,8))
ax[0].plot(monthly.month,monthly.revenue/1e6,color='#1f6feb',label='Ingresos')
ax[0].set_ylabel('M€'); ax[0].set_title('Ingresos mensuales')
ax[1].plot(monthly.month,monthly.ebitda_margin*100,color='#2dd4bf',label='Margen EBITDA')
ax[1].axhline(12,color='#ef4444',ls='--',lw=1,label='Referencia 12%')
ax[1].set_ylabel('%'); ax[1].set_title('Margen EBITDA mensual'); ax[1].legend()
plt.tight_layout(); plt.savefig(FIGURES/'02_pyg_tendencia.png',dpi=180,bbox_inches='tight'); plt.show()

## Lectura ejecutiva

In [4]:
best=annual.loc[annual.ebitda_margin.idxmax()]
print(f"Mejor margen anual: {int(best.year)} con {best.ebitda_margin:.1%}. EBITDA acumulado: {annual.ebitda.sum()/1e6:.1f} M€.")

Mejor margen anual: 2026 con 9.9%. EBITDA acumulado: 39.6 M€.


## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.